# Analisis Keterlambatan Penerbangan AS 2015 dengan PySpark
Midterm Project Big Data & AI. Notebook ini dijalankan berurutan dari atas ke bawah.

**Tahap 1: Setup Spark dan ingestion.** Semua angka (jumlah baris, dll.) dihitung oleh notebook ini, bukan disalin dari draf proposal.

## 1. Konfigurasi Spark
Satu sel konfigurasi: mode lokal memakai semua core, memori driver 8 GB, dan 16 partisi shuffle.

In [ ]:
# Membuat SparkSession dengan konfigurasi tetap (dipakai seluruh notebook)
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("flight_delays_2015")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

# Mengurangi log agar output notebook tidak penuh
spark.sparkContext.setLogLevel("WARN")

## 2. Versi lingkungan
Versi dicatat agar penguji tahu lingkungan yang dipakai.

In [ ]:
# Mencetak versi Python, PySpark, dan Java, serta konfigurasi Spark penting
import sys
import pyspark

print("Python  :", sys.version)
print("PySpark :", pyspark.__version__)
print("Java    :", spark._jvm.System.getProperty("java.version"))
print("master  :", spark.sparkContext.master)
print("driver.memory      :", spark.conf.get("spark.driver.memory"))
print("shuffle.partitions :", spark.conf.get("spark.sql.shuffle.partitions"))

## 3. Membaca data dengan schema eksplisit
Schema ditulis sendiri (tanpa inferSchema). Kolom jam format HHMM dibaca sebagai **string** agar nol di depan (misalnya `0005`) tidak hilang.

In [ ]:
# Mengimpor tipe data untuk membuat schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Schema flights.csv (31 kolom, urutan sama dengan header file)
schema_flights = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("AIRLINE", StringType(), True),
    StructField("FLIGHT_NUMBER", IntegerType(), True),
    StructField("TAIL_NUMBER", StringType(), True),
    StructField("ORIGIN_AIRPORT", StringType(), True),
    StructField("DESTINATION_AIRPORT", StringType(), True),
    StructField("SCHEDULED_DEPARTURE", StringType(), True),   # HHMM, string
    StructField("DEPARTURE_TIME", StringType(), True),        # HHMM, string
    StructField("DEPARTURE_DELAY", IntegerType(), True),
    StructField("TAXI_OUT", IntegerType(), True),
    StructField("WHEELS_OFF", StringType(), True),            # HHMM, string
    StructField("SCHEDULED_TIME", IntegerType(), True),
    StructField("ELAPSED_TIME", IntegerType(), True),
    StructField("AIR_TIME", IntegerType(), True),
    StructField("DISTANCE", IntegerType(), True),
    StructField("WHEELS_ON", StringType(), True),             # HHMM, string
    StructField("TAXI_IN", IntegerType(), True),
    StructField("SCHEDULED_ARRIVAL", StringType(), True),     # HHMM, string
    StructField("ARRIVAL_TIME", StringType(), True),          # HHMM, string
    StructField("ARRIVAL_DELAY", IntegerType(), True),
    StructField("DIVERTED", IntegerType(), True),
    StructField("CANCELLED", IntegerType(), True),
    StructField("CANCELLATION_REASON", StringType(), True),
    StructField("AIR_SYSTEM_DELAY", IntegerType(), True),
    StructField("SECURITY_DELAY", IntegerType(), True),
    StructField("AIRLINE_DELAY", IntegerType(), True),
    StructField("LATE_AIRCRAFT_DELAY", IntegerType(), True),
    StructField("WEATHER_DELAY", IntegerType(), True),
])

In [ ]:
# Schema airports.csv
schema_airports = StructType([
    StructField("IATA_CODE", StringType(), True),
    StructField("AIRPORT", StringType(), True),
    StructField("CITY", StringType(), True),
    StructField("STATE", StringType(), True),
    StructField("COUNTRY", StringType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
])

In [ ]:
# Schema airlines.csv
schema_airlines = StructType([
    StructField("IATA_CODE", StringType(), True),
    StructField("AIRLINE", StringType(), True),
])

In [ ]:
# Membaca flights.csv memakai schema eksplisit
df_flights = spark.read.csv("../data/flights.csv", header=True, schema=schema_flights)

In [ ]:
# Membaca airports.csv
df_airports = spark.read.csv("../data/airports.csv", header=True, schema=schema_airports)

In [ ]:
# Membaca airlines.csv
df_airlines = spark.read.csv("../data/airlines.csv", header=True, schema=schema_airlines)

## 4. Schema
Memastikan tipe kolom sesuai rencana (kolom jam bertipe string).

In [ ]:
# Schema flights
df_flights.printSchema()

In [ ]:
# Schema airports
df_airports.printSchema()

In [ ]:
# Schema airlines
df_airlines.printSchema()

## 5. Jumlah baris dan jumlah partisi
Jumlah baris dihitung dari data, lalu dibandingkan dengan dokumentasi sumber saat laporan disusun. Jumlah partisi menunjukkan bagaimana Spark membagi file saat dibaca.

In [ ]:
# Jumlah baris dan kolom tiap tabel
print("flights  :", df_flights.count(), "baris,", len(df_flights.columns), "kolom")
print("airports :", df_airports.count(), "baris,", len(df_airports.columns), "kolom")
print("airlines :", df_airlines.count(), "baris,", len(df_airlines.columns), "kolom")

In [ ]:
# Jumlah partisi saat data dibaca
print("partisi flights  :", df_flights.rdd.getNumPartitions())
print("partisi airports :", df_airports.rdd.getNumPartitions())
print("partisi airlines :", df_airlines.rdd.getNumPartitions())

## 6. Sampel data
Data berupa catatan per penerbangan (bukan per penumpang), jadi tidak ada data pribadi. Sampel kecil ditampilkan untuk memeriksa bahwa kolom jam masih berformat HHMM dengan nol di depan.

In [ ]:
# Lima baris pertama flights (kolom penting saja agar mudah dibaca)
df_flights.select("YEAR", "MONTH", "DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT",
                  "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE", "DEPARTURE_TIME",
                  "ARRIVAL_DELAY", "CANCELLED", "DIVERTED").show(5)

In [ ]:
# Lima baris pertama airports
df_airports.show(5, truncate=False)

In [ ]:
# Seluruh airlines (hanya sedikit baris)
df_airlines.show(truncate=False)

## 7. Menyimpan flights ke Parquet
CSV mentah tetap di `../data/` sebagai lapisan mentah. Salinan Parquet (kolumnar, schema tersimpan) disimpan di `../data/processed/flights` untuk dipakai tahap berikutnya. Mode `overwrite` supaya sel aman dijalankan ulang.

In [ ]:
# Menulis flights ke Parquet (folder data/processed dibuat otomatis oleh Spark)
df_flights.write.mode("overwrite").parquet("../data/processed/flights")

In [ ]:
# Verifikasi: baca ulang Parquet, jumlah baris harus sama dengan CSV
df_cek = spark.read.parquet("../data/processed/flights")
print("baris Parquet:", df_cek.count())
print("partisi Parquet:", df_cek.rdd.getNumPartitions())